In [17]:
import numpy as np

In [4]:
import pandas as pd

df = pd.read_csv('../data/raw/results.csv')
print(df.shape)
df.head()

(49477, 9)


,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
0,1872-11-30,Scotland,England,0.0,0.0,Friendly,Glasgow,Scotland,False
1,1873-03-08,England,Scotland,4.0,2.0,Friendly,London,England,False
2,1874-03-07,Scotland,England,2.0,1.0,Friendly,Glasgow,Scotland,False
3,1875-03-06,England,Scotland,2.0,2.0,Friendly,London,England,False
4,1876-03-04,Scotland,England,3.0,0.0,Friendly,Glasgow,Scotland,False


In [5]:
df.dtypes

date              str
home_team         str
away_team         str
home_score    float64
away_score    float64
tournament        str
city              str
country           str
neutral          bool
dtype: object

In [6]:
df['home_score'].isna().sum()

np.int64(52)

In [7]:
df['away_score'].isna().sum()

np.int64(52)

In [8]:
df[df['home_score'].isna()]

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49425,2026-06-17,Portugal,DR Congo,NaN,NaN,FIFA World Cup,Houston,United States,True
49426,2026-06-17,Uzbekistan,Colombia,NaN,NaN,FIFA World Cup,Mexico City,Mexico,True
49427,2026-06-17,England,Croatia,NaN,NaN,FIFA World Cup,Arlington,United States,True
49428,2026-06-17,Ghana,Panama,NaN,NaN,FIFA World Cup,Toronto,Canada,True
49429,2026-06-18,Czech Republic,South Africa,NaN,NaN,FIFA World Cup,Atlanta,United States,True
49430,2026-06-18,Mexico,South Korea,NaN,NaN,FIFA World Cup,Zapopan,Mexico,False
49431,2026-06-18,Switzerland,Bosnia and Herzegovina,NaN,NaN,FIFA World Cup,Inglewood,United States,True
49432,2026-06-18,Canada,Qatar,NaN,NaN,FIFA World Cup,Vancouver,Canada,False
49433,2026-06-19,Scotland,Morocco,NaN,NaN,FIFA World Cup,Foxborough,United States,True
49434,2026-06-19,Brazil,Haiti,NaN,NaN,FIFA World Cup,Philadelphia,United States,True


In [9]:
df['date'] = pd.to_datetime(df['date'])

In [10]:
df.dtypes

date          datetime64[us]
home_team                str
away_team                str
home_score           float64
away_score           float64
tournament               str
city                     str
country                  str
neutral                 bool
dtype: object

In [11]:
partidos_a_predecir = df[df['home_score'].isna()].copy()
df_historico = df[df['home_score'].notna()].copy()

print(partidos_a_predecir.shape)
print(df_historico.shape)

(52, 9)
(49425, 9)


In [12]:
df_historico.dtypes

date          datetime64[us]
home_team                str
away_team                str
home_score           float64
away_score           float64
tournament               str
city                     str
country                  str
neutral                 bool
dtype: object

In [13]:
partidos_argentina = df_historico[(df_historico['home_team'] == 'Argentina') | (df_historico['away_team'] == 'Argentina')]

In [14]:
print(partidos_argentina.shape)

(1070, 9)


In [15]:
partidos_argentina.tail()

,date,home_team,away_team,home_score,away_score,tournament,city,country,neutral
49147,2026-03-27,Argentina,Mauritania,2.0,1.0,Friendly,Buenos Aires,Argentina,False
49217,2026-03-31,Argentina,Zambia,5.0,0.0,Friendly,Buenos Aires,Argentina,False
49341,2026-06-06,Argentina,Honduras,2.0,0.0,Friendly,College Station,United States,True
49380,2026-06-09,Argentina,Iceland,3.0,0.0,Friendly,Auburn,United States,True
49423,2026-06-16,Argentina,Algeria,3.0,0.0,FIFA World Cup,Kansas City,United States,True


In [18]:
partidos_argentina['goles_argentina'] = np.where(
    partidos_argentina['home_team'] == 'Argentina',  # condición
    partidos_argentina['home_score'],                 # si es local → tomá home_score
    partidos_argentina['away_score']                  # si es visitante → tomá away_score
)

In [21]:
partidos_argentina['goles_argentina'].mean()

np.float64(1.8981308411214954)

In [30]:
goles_local = df_historico.groupby('home_team')['home_score'].sum()
goles_visitante = df_historico.groupby('away_team')['away_score'].sum()

goles_totales = goles_local.add(goles_visitante, fill_value=0)

In [31]:
print(goles_totales.sort_values(ascending=False).head(10))

home_team
England        2381.0
Germany        2327.0
Brazil         2306.0
Sweden         2176.0
Argentina      2031.0
Hungary        2011.0
Netherlands    1843.0
South Korea    1794.0
Mexico         1769.0
France         1718.0
dtype: float64


In [33]:
partidos_local = df_historico.groupby('home_team').size()
partidos_visitante = df_historico.groupby('away_team').size()
partidos_totales = partidos_local.add(partidos_visitante, fill_value=0)

print(partidos_totales.sort_values(ascending=False).head(10))

home_team
Sweden         1102.0
England        1090.0
Argentina      1070.0
Brazil         1060.0
Germany        1032.0
South Korea    1008.0
Hungary        1006.0
Mexico         1004.0
Uruguay         971.0
France          936.0
dtype: float64


In [36]:
promedio_goles_totales = goles_totales / partidos_totales

print(promedio_goles_totales.sort_values(ascending=False).head(10))

home_team
Quebec                8.000000
Elba Island           4.500000
Yorkshire             3.857143
Parishes of Jersey    3.666667
Cascadia              3.285714
Isle of Man           3.206897
Provence              3.173913
Occitania             3.121212
Sápmi                 3.103448
East Turkestan        3.000000
dtype: float64


In [37]:
promedio_goles_totales[partidos_totales >= 50].sort_values(ascending=False).head(10)

home_team
Isle of Man       3.206897
Jersey            2.744681
Tahiti            2.714876
New Caledonia     2.633962
Guernsey          2.566667
Basque Country    2.562500
Fiji              2.268657
Germany           2.254845
England           2.184404
Brazil            2.175472
dtype: float64

In [40]:
condiciones = [
    df_historico['home_score'] > df_historico['away_score'],
    df_historico['home_score'] == df_historico['away_score'],
    df_historico['home_score'] < df_historico['away_score']
]

valores = ['home_win', 'draw', 'away_win']

df_historico['resultado'] = np.select(condiciones, valores, default='draw')

In [41]:
df_historico['resultado'].value_counts()

resultado
home_win    24222
away_win    13962
draw        11241
Name: count, dtype: int64